# Ecommerce Customers Executable Dashboard

This notebook is a reproducible replacement for the original university Looker Studio dashboard, whose datasource is no longer available.

It uses the same `Ecommerce Customers` dataset already preserved in this project and supports two loading modes:

- local dashboard CSV from `dashboard/data/Ecommerce Customers.csv`
- local project CSV from `data/Ecommerce Customers.csv`
- Databricks table `default.ecommerce_customers`

The dashboard is built with real fields only:

- `Email`
- `Address`
- `Avatar`
- `Avg. Session Length`
- `Time on App`
- `Time on Website`
- `Length of Membership`
- `Yearly Amount Spent`


## 1. Title and project context

This dashboard preserves the visualization objective of the original course project while removing the dependency on the broken Looker Studio datasource.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
}

missing_packages = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All base dashboard packages are already installed.")


In [ ]:
import importlib.util
import subprocess
import sys

CELL_REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
}

cell_missing_packages = [pip_name for import_name, pip_name in CELL_REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]

if cell_missing_packages:
    print("Installing missing packages for this notebook cell:", ", ".join(cell_missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *cell_missing_packages])

from pathlib import Path
from typing import Optional
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    PYSPARK_AVAILABLE = True
except Exception:
    SparkSession = None
    F = None
    PYSPARK_AVAILABLE = False


## 2. Dataset loading


In [ ]:
EXPECTED_COLUMNS = [
    "Email",
    "Address",
    "Avatar",
    "Avg. Session Length",
    "Time on App",
    "Time on Website",
    "Length of Membership",
    "Yearly Amount Spent",
]

def find_dataset_path() -> Optional[Path]:
    notebook_root = Path.cwd()
    search_roots = [notebook_root, *notebook_root.parents]
    candidates = [
        notebook_root / "data" / "Ecommerce Customers.csv",
    ]
    for root in search_roots:
        candidates.append(root / "data" / "Ecommerce Customers.csv")
        candidates.append(root / "Ecommerce Customers.csv")
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def get_or_create_spark():
    if not PYSPARK_AVAILABLE:
        return None
    if "spark" in globals() and spark is not None:
        return spark
    try:
        return SparkSession.builder.master("local[*]").appName("ecommerce-dashboard").getOrCreate()
    except Exception:
        return None

csv_path = find_dataset_path()
spark_session = get_or_create_spark()
source_mode = None
spark_df = None
df = None

if csv_path is not None and spark_session is not None:
    spark_df = spark_session.read.option("header", True).option("inferSchema", True).csv(str(csv_path))
    source_mode = f"local_csv_via_spark: {csv_path}"
elif csv_path is not None:
    df = pd.read_csv(csv_path)
    source_mode = f"local_csv_via_pandas: {csv_path}"
elif spark_session is not None:
    try:
        spark_df = spark_session.table("default.ecommerce_customers")
        source_mode = "databricks_table: default.ecommerce_customers"
    except Exception as exc:
        raise FileNotFoundError("Dataset not found in local data folder and Databricks table default.ecommerce_customers is unavailable.") from exc
else:
    raise FileNotFoundError("Dataset not found in local data folder and PySpark/Databricks table access is unavailable.")

if spark_df is not None:
    missing_columns = [c for c in EXPECTED_COLUMNS if c not in spark_df.columns]
    if missing_columns:
        raise ValueError(f"Dataset is missing expected columns: {missing_columns}")
    df = spark_df.select(*EXPECTED_COLUMNS).toPandas()
else:
    missing_columns = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Dataset is missing expected columns: {missing_columns}")
    df = df[EXPECTED_COLUMNS].copy()

for col in [
    "Avg. Session Length",
    "Time on App",
    "Time on Website",
    "Length of Membership",
    "Yearly Amount Spent",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded dataset using: {source_mode}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head()


## 3. Data quality check


In [ ]:
quality_summary = pd.DataFrame({
    "column": df.columns,
    "missing_values": [int(df[c].isna().sum()) for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns],
})

duplicate_emails = int(df.duplicated(subset=["Email"]).sum())
duplicate_full_rows = int(df.duplicated().sum())

print("Duplicate emails:", duplicate_emails)
print("Duplicate full rows:", duplicate_full_rows)
quality_summary


## 4. KPI computation


In [ ]:
kpis = {
    "Average Yearly Amount Spent": float(df["Yearly Amount Spent"].mean()),
    "Average Time on App": float(df["Time on App"].mean()),
    "Average Time on Website": float(df["Time on Website"].mean()),
    "Average Length of Membership": float(df["Length of Membership"].mean()),
    "Number of Customers": int(df["Email"].nunique()),
}

kpi_table = pd.DataFrame({"KPI": list(kpis.keys()), "Value": list(kpis.values())})
kpi_table


## 5. KPI display


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
fig.suptitle("Ecommerce Customers KPI Overview", fontsize=18, fontweight="bold", y=1.08)

kpi_formats = {
    "Average Yearly Amount Spent": "${:,.2f}",
    "Average Time on App": "{:.2f}",
    "Average Time on Website": "{:.2f}",
    "Average Length of Membership": "{:.2f}",
    "Number of Customers": "{:,.0f}",
}

for ax, (label, value) in zip(axes, kpis.items()):
    ax.axis("off")
    ax.set_facecolor("#f7f9fc")
    ax.text(0.5, 0.68, label, ha="center", va="center", fontsize=11, color="#334155", wrap=True)
    ax.text(0.5, 0.35, kpi_formats[label].format(value), ha="center", va="center", fontsize=18, fontweight="bold", color="#0f172a")

plt.tight_layout()
plt.show()


## 6. Chart 1: Top 10 customers by spending


In [ ]:
top_10_customers = df[["Email", "Yearly Amount Spent"]].sort_values("Yearly Amount Spent", ascending=False).head(10)

plt.figure(figsize=(11, 5.5))
plt.barh(top_10_customers["Email"], top_10_customers["Yearly Amount Spent"], color="#0f766e")
plt.gca().invert_yaxis()
plt.title("Top 10 Customers by Yearly Amount Spent", fontsize=15, fontweight="bold")
plt.xlabel("Yearly Amount Spent")
plt.ylabel("Email")
plt.tight_layout()
plt.show()

top_10_customers


## 7. Chart 2: Membership length vs spending


In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(df["Length of Membership"], df["Yearly Amount Spent"], alpha=0.65, color="#2563eb", edgecolors="none")
plt.title("Length of Membership vs Yearly Amount Spent", fontsize=15, fontweight="bold")
plt.xlabel("Length of Membership")
plt.ylabel("Yearly Amount Spent")
plt.tight_layout()
plt.show()


## 8. Chart 3: Spending distribution


In [ ]:
plt.figure(figsize=(9, 5.5))
plt.hist(df["Yearly Amount Spent"].dropna(), bins=25, color="#7c3aed", edgecolor="white")
plt.title("Distribution of Yearly Amount Spent", fontsize=15, fontweight="bold")
plt.xlabel("Yearly Amount Spent")
plt.ylabel("Customer Count")
plt.tight_layout()
plt.show()


## 9. Chart 4: Average spending by membership range


In [ ]:
membership_bins = [0, 1, 2, 3, 4, 5, np.inf]
membership_labels = ["0-1", "1-2", "2-3", "3-4", "4-5", "5+"]

df_membership = df.copy()
df_membership["Membership Range"] = pd.cut(
    df_membership["Length of Membership"],
    bins=membership_bins,
    labels=membership_labels,
    include_lowest=True,
)

membership_spending = (
    df_membership.groupby("Membership Range", observed=False)["Yearly Amount Spent"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9, 5.5))
plt.bar(membership_spending["Membership Range"].astype(str), membership_spending["Yearly Amount Spent"], color="#ea580c")
plt.title("Average Yearly Amount Spent by Membership Range", fontsize=15, fontweight="bold")
plt.xlabel("Membership Range")
plt.ylabel("Average Yearly Amount Spent")
plt.tight_layout()
plt.show()

membership_spending


## 10. Chart 5: Average app time vs average website time


In [ ]:
platform_avg = pd.DataFrame({
    "Platform": ["App", "Website"],
    "Average Time": [
        df["Time on App"].mean(),
        df["Time on Website"].mean(),
    ],
})

plt.figure(figsize=(7, 5))
plt.bar(platform_avg["Platform"], platform_avg["Average Time"], color=["#0891b2", "#14b8a6"])
plt.title("Average Time on App vs Average Time on Website", fontsize=15, fontweight="bold")
plt.ylabel("Average Time")
plt.tight_layout()
plt.show()

platform_avg


## 11. Optional chart: spending range distribution


In [ ]:
optional_spending_distribution = None

try:
    temp = df[["Yearly Amount Spent"]].dropna().copy()
    temp["Fascia_spesa"] = pd.qcut(
        temp["Yearly Amount Spent"],
        q=4,
        labels=["Low", "Medium", "High", "Very High"],
        duplicates="drop",
    )
    optional_spending_distribution = temp["Fascia_spesa"].value_counts().sort_index()

    plt.figure(figsize=(8, 5))
    plt.bar(optional_spending_distribution.index.astype(str), optional_spending_distribution.values, color="#be123c")
    plt.title("Optional Spending Range Distribution", fontsize=15, fontweight="bold")
    plt.xlabel("Spending Range")
    plt.ylabel("Customer Count")
    plt.tight_layout()
    plt.show()

    display(optional_spending_distribution.rename("Customer Count").to_frame())
except Exception as exc:
    print("Optional spending range chart skipped:", exc)


## 12. Final business insights


In [ ]:
overall_avg_spending = df["Yearly Amount Spent"].mean()
long_session_avg_spending = df.loc[df["Avg. Session Length"] > 34, "Yearly Amount Spent"].mean()
leading_platform = "App" if df["Time on App"].mean() > df["Time on Website"].mean() else "Website"
top_customer = top_10_customers.iloc[0]

insights = [
    f"Average yearly amount spent is ${overall_avg_spending:,.2f} across {df['Email'].nunique():,} customers.",
    f"The higher-engagement channel by average usage in this dataset is {leading_platform}.",
    f"The highest-value observed customer in the dataset is {top_customer['Email']} with yearly spending of ${top_customer['Yearly Amount Spent']:,.2f}.",
    f"Customers with Avg. Session Length > 34 have an average yearly spend of ${long_session_avg_spending:,.2f} based on the real filtered data.",
    "Membership length is visualized as a key explanatory variable because both the original Databricks enrichment notebook and this dashboard show it as an important analytical axis.",
]

for idx, item in enumerate(insights, start=1):
    print(f"{idx}. {item}")


## 13. Reproducibility notes

- This notebook does not require credentials.
- It does not depend on the broken Looker Studio datasource.
- It prefers the local project CSV when available.
- If the local CSV is unavailable and Spark is available, it can load `default.ecommerce_customers` in Databricks.
- Visualizations are generated from real loaded data at runtime.
- No fake metrics or hardcoded dashboard values are used.
